# Gemma attention memory test

One notebook for issue 569. It runs meeting `45Wohe5D1-A` twice on Gemma 4 31B in 4-bit, then stops. The first run uses the default attention. The second uses sdpa. Each run writes its files to Google Drive and appends its peak-memory line there too.

The owner approved this run on 2026-11-19 after the Colab tab printed `NVIDIA A100-SXM4-80GB  80.0 GB`.

**Where to run it.** Opened from GitHub. Choose **Runtime → Run all**. Leave every other Colab tab alone.

If the first code cell stops with `STOP: this GPU has 40 GB`, the upload attached the wrong card. **Runtime → Change runtime type → A100**, run the first code cell again, and continue only when it prints `OK: about 80 GB`.


In [ ]:
# 1. Stop unless this runtime is an A100 of about 80 GB.
import subprocess

info = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=True).stdout.strip()
name, mem = [part.strip() for part in info.split(",")]
gb = float(mem) / 1024
print(f"{name}  {gb:.1f} GB")
if "A100" not in name.upper() or gb < 70:
    raise SystemExit(
        f"STOP: this GPU is {name} with {gb:.0f} GB, not an A100 of about 80 GB. "
        "Delete this runtime and attach an A100 again. Do not reconnect the other project's tab.")
print("OK: about 80 GB.")


In [ ]:
# 2. Get the code from main and install it.
import os
import subprocess
import sys

REPO = "/content/open-navigator"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/getcommunityone/open-navigator.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=5.5.0", "accelerate>=1.0",
                "bitsandbytes>=0.45", "databricks-sdk", "loguru", "pydantic>=2", "pyyaml"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", f"{REPO}/packages/llm"], check=True)
sys.path.insert(0, f"{REPO}/packages/llm/src")
os.environ.setdefault("LLM_USAGE_LOG_DIR", "/content/llm_usage")

import transformers
from llm.gpu import colab_policy_analyze as cpa
print("transformers", transformers.__version__, "| attn flag present:",
      "--attn-implementation" in cpa.build_parser().format_help())


In [ ]:
# 3. Mount Drive. Refuse Databricks, so nothing is written only on this runtime's disk.
import os
from pathlib import Path

from llm.gpu import colab_io

if colab_io.get_secret("DATABRICKS_HOST") and colab_io.get_secret("DATABRICKS_TOKEN"):
    raise SystemExit(
        "STOP: DATABRICKS_HOST and DATABRICKS_TOKEN are on for this notebook, so results would go to "
        "Databricks instead of Drive. Turn off notebook access for both secrets, then run this cell again.")
hf = colab_io.get_secret("HF_TOKEN")
if not hf:
    raise SystemExit("STOP: add an HF_TOKEN secret (a read token whose account accepted the Gemma licence), then rerun.")
os.environ["HF_TOKEN"] = hf

from google.colab import drive
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/open-navigator/crawl")
ROOT.mkdir(parents=True, exist_ok=True)
print("Writing to", ROOT)


Set the compute-unit balance from Colab's **Resources** panel in the next cell if you have it open. Leaving `0` still runs the test. The cost line will say the balance was not entered.


In [ ]:
# 4. Usage before. Type the balance from the Resources panel. 0 means not entered.
cu_balance_before = 0.0  #@param {type:"number"}
cu_per_hour = 0.0  #@param {type:"number"}
from datetime import datetime, timezone
print(f"usage before: {datetime.now(timezone.utc).isoformat(timespec='seconds')} gpu=A100-80GB "
      f"cu_balance={cu_balance_before or 'not entered'} cu_per_hour={cu_per_hour or 'not entered'}")


In [ ]:
# 5. The one meeting, on Drive, and a file that collects the two peak-memory lines.
from pathlib import Path

REQ = ROOT / "policy_export/2026-11-17/requests/45Wohe5D1-A.json"
if not REQ.is_file():
    raise SystemExit(
        "STOP: the meeting file is not on Drive yet.\n"
        "On the laptop it is data/cache/colab_gemma_export/2026-11-17/requests/45Wohe5D1-A.json\n"
        "Upload that file to MyDrive/open-navigator/crawl/policy_export/2026-11-17/requests/45Wohe5D1-A.json\n"
        "Then run this cell again. Do not start the model before this file is there.")
ONE = ROOT / "policy_export/2026-11-17/one_45Wohe5D1-A.txt"
ONE.write_text("45Wohe5D1-A\n", encoding="utf-8")
PEAK_FILE = ROOT / "attn_memory_test_2026-11-19_peaks.txt"
PEAK_FILE.write_text("", encoding="utf-8")
print("OK:", REQ)
print("Peak lines will be appended to", PEAK_FILE)


In [ ]:
# 6. Shared runner. Results are written under ROOT, which is the Drive folder.
from loguru import logger
from llm.gpu import colab_policy_analyze as cpa

def _peak_sink(message):
    text = message.record["message"]
    if "peak" in text and "attn" in text:
        with PEAK_FILE.open("a", encoding="utf-8") as handle:
            handle.write(text + "\n")

logger.add(_peak_sink, level="INFO")

def run_attention(attn, out_dir):
    argv = [
        "--videos-file", "policy_export/2026-11-17/one_45Wohe5D1-A.txt",
        "--model", "google/gemma-4-31B-it",
        "--quantize", "4bit",
        "--limit", "1",
        "--run-date", "2026-11-19",
        "--out", out_dir,
        "--drive-root", str(ROOT),
    ]
    if attn:
        argv += ["--attn-implementation", attn]
    if cu_per_hour:
        argv += ["--cu-per-hour", str(cu_per_hour)]
    print("starting", out_dir, "attn", attn or "default")
    code = cpa.main(argv)
    folder = ROOT / out_dir / "2026-11-19"
    names = sorted(p.name for p in folder.glob("45Wohe5D1-A.*")) if folder.is_dir() else []
    print("Drive files:", names or "NONE — do not disconnect")
    print("exit", code)
    return code


In [ ]:
# 7. Default attention. Wait until this cell finishes before the next one starts.
run_attention(None, "policy_analysis_attn_default")
print("--- peak file so far ---")
print(PEAK_FILE.read_text(encoding="utf-8") or "(no peak line yet)")


In [ ]:
# 8. sdpa. A separate folder, so a success above does not skip this meeting.
run_attention("sdpa", "policy_analysis_attn_sdpa")
print("--- both peak lines ---")
print(PEAK_FILE.read_text(encoding="utf-8") or "(no peak line was written)")
print("Copy the peak file from Drive if you close this tab:")
print(PEAK_FILE)


Read the two lines in the peak file.

- `out of GPU memory` is a failure. `generated ... peak GPU memory` is a fit.
- **Both fail.** sdpa does not make this 56,752-token meeting fit on an 80 GB A100 in 31B 4-bit. Stop.
- **Only sdpa fits.** Later meetings this long need `--attn-implementation sdpa`.
- **Both fit.** The 18 November failure did not come back. The limit is above 56,752 tokens. The 81,957-token meeting is still untested.

A `generated` line means the memory test succeeded even if the answer later fails validation.

Leave this runtime connected until both Drive folders contain `45Wohe5D1-A.analysis.json` or `45Wohe5D1-A.failed.json`, and the peak file has two lines. Disconnecting before that wipes only the runtime disk. The Drive copies stay. Do not disconnect the other project's tab.
